# tbats_jax — A100 panel benchmark (T=1500, where T4 hit the wall)

Goal: test whether an A100 GPU unlocks the regime where **T4 structurally failed** — long-series hourly/half-hourly data (T=1500, periods=(24, 168)) at N≥5000.

**Why T=1500 is the sharper test:**
- Per-series work is 2× T=730, pushing memory and compute harder
- T4 at N=5000 T=1500: per-series degraded to **214 ms vs CPU's 196 ms (91% of CPU — GPU *lost*)**
- Half-hourly utility, 5-min IoT, high-freq telemetry panels are common — this is a real use-case
- Taylor dataset is T=3696, even longer — validates scaling toward production

**T4 reference curve** (measured on T=1500 daily panel, periods=(24, 168), k=(3, 5)):

| N     | T4 per-series | T4 warm wall | CPU equivalent | T4 vs CPU |
|-------|---------------|--------------|----------------|-----------|
|  500  | 93 ms         | 46 s         | 98 s           | **2.12×** |
| 1000  | 104 ms        | 104 s        | 196 s          | 1.88×     |
| 5000  | 214 ms        | 1068 s       | 980 s          | **0.92× (LOSES)** |

**Pick A100:** Runtime → Change runtime type → Hardware: **A100 GPU** → Save. (Colab Pro.)

## 1. Install JAX with CUDA

This works for any NVIDIA GPU runtime (T4, L4, A100, H100). Wrong wheel = silent CPU fallback.

In [ ]:
!pip install -q --upgrade 'jax[cuda12]' optimistix

## 2. Upload `tbats_jax_colab.zip`

In [ ]:
from google.colab import files
uploaded = files.upload()

import zipfile, os
for name in uploaded:
    if name.endswith('.zip'):
        with zipfile.ZipFile(name) as z:
            z.extractall('/content')
        print('extracted to /content/tbats_jax')

assert os.path.isdir('/content/tbats_jax')

In [ ]:
%cd /content/tbats_jax
!pip install -q -e .

## 3. Verify the accelerator

Expect `backend: gpu` and device name containing `A100-SXM4-40GB` (or `A100-PCIe`). If it prints `cpu` or `T4`, change the runtime before continuing.

In [ ]:
import jax
jax.config.update('jax_enable_x64', True)
print('backend :', jax.default_backend())
print('devices :', jax.devices())

from tbats_jax.admissibility import _default_method
print('rho method (auto):', _default_method())  # should be "power" on GPU

# Free GPU memory footprint (A100 has 40 GB, we'll use ~500 MB per 1000 series)
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv 2>/dev/null | head -2

## 4. Baseline: N=1000

Direct T4 comparison (104 ms/series, 104 s wall). CPU: 196 ms/series. On A100 expect ~40–60 ms/series — the 39× FP64 headroom and 5× HBM bandwidth should matter.

In [ ]:
from benchmarks.colab_panel_gpu import run
r1000 = run(N=1000, T=1500)

## 5. The regime T4 lost: N=5000

T4 took 1068 s here (91% of CPU's 980 s extrapolated — lost). Memory footprint at T=1500, N=5000, float64: ~1 GB of y + ~4 GB of autodiff intermediates — fits A100's 40 GB with headroom. If A100 keeps per-series flat at ~50 ms (no bandwidth wall), this should finish in ~250 s → ~4× speedup over CPU right where T4 failed.

In [ ]:
r5000 = run(N=5000, T=1500)

## 6. The headline: N=10000

Ten thousand hourly-T=1500 TBATS models in one fused `vmap`. R `forecast::tbats` sequential cost at this scale: ~2 hours. CPU extrapolation at 196 ms/series: ~33 min. A100 target: **~5-10 min** (~3-7× over CPU). Memory footprint: ~10 GB on 40 GB A100.

In [ ]:
r10000 = run(N=10000, T=1500)

## 7. Stretch (optional): N=20000

20k TBATS models. At T=1500 this needs ~20 GB autodiff intermediates — uses half of A100's 40 GB. Compile will be slow (graph 2× bigger). Skip if short on Colab credits — the N=10000 number above is the main result.

In [ ]:
try:
    r20000 = run(N=20000, T=1500)
except Exception as e:
    r20000 = None
    print(f'N=20000 skipped: {e}')

## 8. Summary — A100 vs CPU vs R at T=1500

Speedup is relative to:
- **Apple Silicon M-series CPU** (196 ms/series measured at N=500 T=1500, assumed linear in N)
- **R `forecast::tbats` sequential fixed-k** (343 ms/series measured at N=32 T=1500 → ~740 ms/series including Rscript startup cost at larger N; here the *pure fit* time per series is used)

In [ ]:
cpu_per_series_ms = 196.0  # Apple Silicon M-series CPU, measured at T=1500
r_per_series_ms   = 343.0  # R forecast::tbats fixed-k, pure fit time

backend_str = f'{jax.default_backend().upper()} — {jax.devices()[0]}'
print(f'accelerator: {backend_str}')
print()
hdr = f'{"N":>6} | {"compile":>9} | {"warm":>8} | {"ms/ser":>7} | {"CPU eq":>10} | {"R eq":>10} | {"vs CPU":>8} | {"vs R":>7}'
print(hdr)
print('-' * len(hdr))
rows = [(1000, r1000), (5000, r5000), (10000, r10000)]
if r20000 is not None:
    rows.append((20000, r20000))
for N, r in rows:
    cpu_eq = N * cpu_per_series_ms / 1000
    r_eq   = N * r_per_series_ms / 1000
    sp_cpu = cpu_eq / r['warm_wall']
    sp_r   = r_eq / r['warm_wall']
    print(f'{N:>6} | {r["compile_time"]:>8.1f}s | {r["warm_wall"]:>7.2f}s | {r["per_series_ms"]:>6.2f} | {cpu_eq:>9.1f}s | {r_eq:>9.1f}s | {sp_cpu:>7.2f}x | {sp_r:>6.1f}x')

# T4 reference for direct comparison
print()
print('T4 reference (same workload, already measured):')
print(f'{"N":>6} | {"warm":>8} | {"ms/ser":>7}')
print('-' * 30)
for N, wall, mps in [(1000, 104.05, 104.05), (5000, 1068.0, 214.0)]:
    print(f'{N:>6} | {wall:>7.2f}s | {mps:>6.2f}')